# JSON Basics — 02: Writing JSON Files

Writing JSON from Spark — controlling output format, compression,
and structure.

Topics: spark.write.json, compression, date formatting, single file output,
write modes, converting to NDJSON vs pretty-printed.


In [ ]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/json_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('json-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

## Step 1 — Basic Write



In [ ]:

import random, datetime
random.seed(42); N=50_000
df_src = spark.range(N).select(
    F.col("id").alias("order_id"),
    (F.col("id")%1000+1).alias("customer_id"),
    F.element_at(F.array([F.lit(c) for c in ["Electronics","Books","Clothing","Food"]]), (F.col("id")%4+1).cast("int")).alias("category"),
    F.round(F.rand(42)*500+5,2).alias("price"),
    F.when(F.col("id")%7==0,F.lit(None)).otherwise(F.round(F.rand(43)*10,1)).alias("score"),
    F.lit(datetime.date.today()).alias("created_date"),
)
OUT = f"{DATA_DIR}/write_test"
df_src.write.mode("overwrite").json(f"{OUT}/default")
import glob as glib
files = glib.glob(f"{OUT}/default/*.json")
size_kb = sum(pathlib.Path(f).stat().st_size for f in files)//1024
print(f"Default write: {len(files)} file(s), {size_kb} KB")
print("Each file is NDJSON — one JSON object per line, no trailing comma")


## Step 2 — Compression



In [ ]:

codecs = ["none", "gzip", "bz2", "deflate", "lz4"]
results = {}
for codec in codecs:
    path = f"{OUT}/codec_{codec}"
    t0=time.time(); df_src.write.mode("overwrite").option("compression",codec).json(path); t_w=time.time()-t0
    ext = ".json.gz" if codec=="gzip" else ".json.bz2" if codec=="bz2" else ".json"
    files = glib.glob(f"{path}/*{ext}") or glib.glob(f"{path}/*.json*")
    mb = sum(pathlib.Path(f).stat().st_size for f in files)/1024/1024
    results[codec] = {"write":t_w,"size":mb}

base = results["none"]["size"]
print(f"{'Codec':<10} {'Write':>8} {'MB':>8} {'vs none':>10}")
print("-"*42)
for c,r in results.items():
    print(f"  {c:<8} {r['write']:>6.2f}s {r['size']:>6.1f} MB {r['size']/base:>9.2f}x")
print()
print("JSON text compresses very well (3-10x) due to repeated field names and values")


## Step 3 — Date and Timestamp Formatting



In [ ]:

import datetime as dt
df_ts = spark.createDataFrame([
    (1, datetime.date(2024,1,15),   dt.datetime(2024,1,15,10,30,0),   "confirmed"),
    (2, datetime.date(2024,2,1),    dt.datetime(2024,2,1,14,0,22),    "shipped"),
    (3, datetime.date(2024,3,20),   dt.datetime(2024,3,20,9,15,30),   "delivered"),
], ["id","order_date","order_ts","status"])

# Default formats
df_ts.write.mode("overwrite").json(f"{OUT}/ts_default")
print("Default date/timestamp output:")
with open(glib.glob(f"{OUT}/ts_default/*.json")[0]) as f: [print(" ",l.strip()) for l in f]

# Custom formats
df_ts.write.mode("overwrite") \
    .option("dateFormat","dd/MM/yyyy") \
    .option("timestampFormat","yyyy-MM-dd HH:mm:ss") \
    .json(f"{OUT}/ts_custom")
print("\nCustom date/timestamp format:")
with open(glib.glob(f"{OUT}/ts_custom/*.json")[0]) as f: [print(" ",l.strip()) for l in f]


## Step 4 — Single File Output



In [ ]:

# coalesce(1) → single file (for tools expecting one file)
df_src.coalesce(1).write.mode("overwrite").json(f"{OUT}/single")
files = glib.glob(f"{OUT}/single/*.json")
print(f"Single file: {len(files)} file(s)")
size_kb = sum(pathlib.Path(f).stat().st_size for f in files)//1024
print(f"Size: {size_kb} KB")

# Show first 3 lines
with open(files[0]) as f:
    lines = [f.readline() for _ in range(3)]
print("\nFirst 3 lines (NDJSON format):")
for l in lines: print(" ", l.strip())


## Step 5 — Write Modes



In [ ]:

TABLE = f"{OUT}/modes"
df_src.limit(1000).write.mode("overwrite").json(TABLE)
print(f"overwrite: {spark.read.json(TABLE).count():,} rows")

df_src.limit(200).write.mode("append").json(TABLE)
print(f"append   : {spark.read.json(TABLE).count():,} rows (+200)")

df_src.limit(500).write.mode("ignore").json(TABLE)
print(f"ignore   : {spark.read.json(TABLE).count():,} rows (unchanged)")

try:
    df_src.limit(100).write.json(TABLE)
except Exception as e:
    print(f"error    : {type(e).__name__} (path exists)")


## Summary



In [ ]:

print("""
df.write.mode("overwrite")
   .option("compression",      "gzip")           # none|gzip|bz2|deflate|lz4
   .option("dateFormat",       "yyyy-MM-dd")
   .option("timestampFormat",  "yyyy-MM-dd HH:mm:ss")
   .option("ignoreNullFields", "false")           # False = include null fields
   .json("/path/output/")

Single file:    df.coalesce(1).write.json(path)
Compressed:     df.write.option("compression","gzip").json(path)
NDJSON output:  always — Spark JSON writer is always NDJSON (one object per line)
Pretty-print:   not supported natively — use df.toJSON().collect() + json.dumps(indent=2)
""")
